In [0]:
from pyspark.sql import functions as f 
import os , json 
from dotenv import load_dotenv
load_dotenv()



In [0]:
dbutils.widgets.text("source","")
source_path = dbutils.widgets.get("source")
source_path

In [0]:
orders = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFile.inferSchemaType", "true")
          .option("cloudFiles.schemaLocation",f"{source_path}/schema")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(f"{source_path}/orders")
)


In [0]:
def myfunc(df,batchId):
    #display(df) 
    df.write.format("delta").mode("overwrite").saveAsTable('namaste_catalog.jobs.orders_streaming')
                                                           

In [0]:
(orders.writeStream
 .outputMode("append")
 .foreachBatch(myfunc)
 .trigger(once=True)
 .option("checkpointLocation",f"{source_path}/checkpoint")
.start()
 
)

In [0]:
customers = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFile.inferSchemaType", "true")
          .option("cloudFiles.schemaLocation",f"{source_path}/schema/customers")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(f"{source_path}/customers")
)


In [0]:
from pyspark.sql import DataFrame
def mycustomers(df: DataFrame, batchId :int) -> None:
    # The type of df is pyspark.sql.dataframe.DataFrame
    df.write.format("delta").mode("overwrite").saveAsTable('namaste_catalog.jobs.customers_streaming')


In [0]:
(
 customers.writeStream.
 outputMode("append")
 .foreachBatch(mycustomers)
 .trigger(once=True)
 .option("checkpointLocation", f"{source_path}/checkpoint/customers")
 .start()
 
 )

In [0]:
%skip
%sql
select * from namaste_catalog.jobs.customers_streaming

In [0]:
%skip
%sql
select * from namaste_catalog.jobs.orders_streaming